# 기간 내 현금영수증 조회 및 승인·취소 대사

지정한 단말기의 현금영수증(GBN=2)을 기간 전체 페이지로 조회하고,
검증할 승인번호가 있으면 입력한 기대 성격과 실제 응답(NRSPC_NM / NRSPC_MSG)을 나란히 확인한다.

- 단말기번호는 `.env`의 `SMARTRO_VAN_TERMID`, 인증키는 `SMARTRO_VAN_API_KEY`에서 읽는다.
- `EXPECTED`를 비워 두면 1번(기간 전체 목록)만 의미가 있다.
- 거절 건도 포함해 조회한다(`REJEC_TYPE=1`). HTTP 200이어도 응답 `CODE`가 `"0000"`인지 확인하고 쓴다.
- 소득공제/지출증빙 구분은 VAN 명세에 없어 API만으로 검증할 수 없다. GET 조회만 하며 발급·취소 요청은 하지 않는다.

공개 명세: https://exttran.smilebiz.co.kr/getApiSvcInfoData?SVC_CTGR=VAN

## 0. 설정

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = [parent, parent / "smilebiz-van/van-api"]
    helper_dir = next((p for p in candidates if (p / "van_test_helpers.py").exists()), None)
    if helper_dir is not None:
        sys.path.insert(0, str(helper_dir))
        break
else:
    raise RuntimeError("노트북 폴더 또는 저장소 루트에서 실행하세요.")
from van_test_helpers import fetch_pages, sales_params, env_value

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# -- 조회 조건 --------------------------------------------------
TERMID = env_value("SMARTRO_VAN_TERMID")      # 단말기번호 10자리 (.env)
COMP_NO = env_value("SMARTRO_VAN_COMP_NO")    # (선택) 사업자번호
COMP_IDX = env_value("SMARTRO_VAN_COMP_IDX")
GBN = "2"                                     # 현금영수증
SDATE = "20260903"                            # 조회 시작일 (YYYYMMDD)
EDATE = "20260909"                            # 조회 종료일
# 검증할 승인번호 -> 기대 성격. 비워 두면 1번(기간 전체 목록)만 실행한다.
EXPECTED = {}   # 예: {"승인번호1": "소득공제", "승인번호2": "현금결제취소"}
# ------------------------------------------------------------

if len(TERMID) != 10:
    raise ValueError("TERMID는 10자리여야 합니다. .env의 SMARTRO_VAN_TERMID를 확인하세요.")
for _d in (SDATE, EDATE):
    if len(_d) != 8 or not _d.isdigit():
        raise ValueError("날짜는 YYYYMMDD 8자리여야 합니다.")
if SDATE > EDATE:
    raise ValueError("시작일이 종료일보다 늦습니다.")
print(f"단말기 {TERMID} | 현금영수증 | 기간 {SDATE} ~ {EDATE} | 검증 승인번호 {list(EXPECTED) or '없음'}")

## 1. 기간 내 현금영수증 전체 페이지 조회
승인번호 제한 없이 `SDATE ~ EDATE`의 현금영수증(GBN=2)을 모든 페이지 조회한다. 거절 건도 포함한다.

In [ ]:
rows_all, raw_all = fetch_pages(
    "/V1/sales/getSalesList",
    sales_params(SDATE, EDATE, TERMID, GBN, comp_no=COMP_NO, comp_idx=COMP_IDX),
)
df_all = pd.DataFrame(rows_all)
print(f"{SDATE} ~ {EDATE} 현금영수증 {len(df_all)}건 / 수신 페이지 {len(raw_all)}개")
display(df_all)

## 2. 지정 승인번호별 전체 페이지 조회
`EXPECTED`의 각 승인번호를 개별 조회한다. 한 번호에 승인·취소 등 여러 행이 있으면 모두 표시한다.

In [ ]:
rows_target, status_rows = [], []
for authno in EXPECTED:
    try:
        rows, _ = fetch_pages(
            "/V1/sales/getSalesList",
            sales_params(SDATE, EDATE, TERMID, GBN, authno, COMP_NO, COMP_IDX),
        )
        rows_target.extend(rows)
        status_rows.append({"승인번호": authno, "상태": "조회 완료" if rows else "내역 없음",
                            "건수": len(rows), "오류": ""})
    except RuntimeError as error:
        status_rows.append({"승인번호": authno, "상태": "조회 실패", "건수": None, "오류": str(error)})

df_target = pd.DataFrame(rows_target)
if status_rows:
    display(pd.DataFrame(status_rows))
    display(df_target)
else:
    print("EXPECTED가 비어 있습니다. 특정 승인번호를 검증하려면 0번에서 EXPECTED를 채우세요.")

## 3. 기대값과 실제 승인·취소 결과 대사
입력한 기대 성격과 실제 응답(NRSPC_NM/NRSPC_MSG)을 나란히 둔다. 자동으로 옳고 그름을 단정하지 않는다.

In [ ]:
checks = []
for authno, expected in EXPECTED.items():
    matching = [r for r in rows_target if str(r.get("AUTHNO", "")) == authno]
    for row in matching or [{}]:
        checks.append({
            "승인번호": authno,
            "기대 성격(입력값)": expected,
            "실제 승인구분 NRSPC_NM": row.get("NRSPC_NM"),
            "승인결과 NRSPC_MSG": row.get("NRSPC_MSG"),
            "금액 AMT1": row.get("AMT1"),
            "원거래일자 ORGDATE": row.get("ORGDATE"),
            "일련번호 MTRCNO": row.get("MTRCNO"),
            "비고": "일치하는 행 없음" if not row else "",
        })

if checks:
    display(pd.DataFrame(checks))
    print("VAN 명세에는 소득공제/지출증빙 구분 필드가 없어 소득공제 여부는 API만으로 검증할 수 없습니다.")
    print("두 승인번호가 원거래-취소 관계인지도 자동 단정하지 않습니다. NRSPC_NM/NRSPC_MSG 원본을 직접 비교하세요.")
    if not df_target.empty and "AUTHNO" in df_target.columns:
        print()
        print("승인번호별 전체 반환 필드 (열 = 거래):")
        display(df_target.set_index("AUTHNO").T)
else:
    print("EXPECTED가 비어 있어 대사할 항목이 없습니다.")